# Semantic Search Engine For Excel Files
Many a times we need some particular data and it is very tedious when we have this large pile of excel files and we have to search all the sheets. Its very time-consuming and frustating
Here we have built a semantic search engine for excel files in which:
1. The user has to give an input of keywords that he/she is looking for
2. The app would return the top 3 sheets that contain the most relevant information

In [1]:
#Importing important libraries
import pandas as pd
import numpy as np
import os

In [2]:
#Loading all excel files
files = ["A. IATF16949-QUALITY-MANUAL updated 30.9.25.xls",
        "D.Turtle Chart_All Deptt..xlsx",
        "HR-Work Procedure Updated Issue 8.xlsx",
        "MAINT Work Procedure Issue 8.xls",
        "P .List of Customers & CSR matrix.xlsx",
        "Purchase Work Procedure Issue 8.xlsx",
        "Store Work Procedure Issue 8.xls",
        "WP Development .Issue 08.xls",
        "WP Production  Issue 8 (2).xls",
        "WP QAD updated issue 8.xls",
        "WP QMS Issue 8.xls"]

In [3]:
#Writing a loop to check the names of different sheets
for f in files:
    if f.endswith(".xls"):
        xls = pd.ExcelFile(f, engine='xlrd')
    else:  # .xlsx
        xls = pd.ExcelFile(f, engine='openpyxl')
    
    print(f"File: {f}")
    print("Sheets:", xls.sheet_names)
    print("-"*40)

File: A. IATF16949-QUALITY-MANUAL updated 30.9.25.xls
Sheets: ['Approval', '1.0', '2A', '2B', '2C', '3-0 ', '4-0', 'WP-HR-D', '6A-CEO', '6B-COO', '6C-HR', '6D-PUR', '6E-QAD', '6F-PRD', '6G-MAINT', '6H-DVP', '6I-QMS', '6J-PPC', '6K-STORE', '7A', '7B Obsolete', '7B New', '8', '9-1', '10A', '10 B', 'CSR Rev 1', '11', '12']
----------------------------------------
File: D.Turtle Chart_All Deptt..xlsx
Sheets: ['PRD', 'HR', 'PUR-1', 'PUR-2', 'PUR-3', 'PUR-4', 'QAD-1', 'QAD-2', 'QAD-3', 'MAINT-1', 'MAINT-2', 'QMS', 'DVP', 'PPC', 'STR']
----------------------------------------
File: HR-Work Procedure Updated Issue 8.xlsx
Sheets: ['WP-HR-A ', 'WP-HR-B', 'WP-HR-C', 'WP-HR-D1', 'HR-E', 'WP-HR 01', 'WP-HR-02', 'WP-HR-03', 'AUDITOR.COMPETENCE .REVISED', 'CM-Unit I (IATF)', 'CM-Unit II', 'CM-Unit III', 'CM-JSR', 'UNIT 5', 'MR', 'TNA DESCRIPTION', 'PLAN 2021-22', 'Attendence Sheet', 'Training Feedback Form', 'Test', 'Effectiveness eval SPC ', 'Performance Appraisal ', 'Safety - Incident Report', 'On 

In [4]:
#Loading all the files with all sheets, we are using openpyxl and xlrd because some have .xls and .xlxs
import openpyxl
import xlrd
data = []

for file in files:
    sheets = pd.read_excel(file, sheet_name=None)  # loads ALL sheets
    
    for sheet_name, df in sheets.items():
        data.append({
            "file": file,
            "sheet": sheet_name,
            "dataframe": df
        })

print(len(data))

267


In [5]:
#Each item consists of a dataframe consisting the data and the metadata. Document stores all the processed data

documents = []

for item in data:
    df = item["dataframe"].fillna("").astype(str) #cleaning the data

    text = "\n".join(df.agg(" | ".join, axis=1)) #turning the rows into text

    documents.append({
        "text": text,
        "file": item["file"],
        "sheet": item["sheet"]
    })

In [6]:
#Here we are checking which are the sheets that contain charts and shapes
import xlwings as xw

file = "A. IATF16949-QUALITY-MANUAL updated 30.9.25.xls"

app = xw.App(visible=False)
wb = app.books.open(file)

for sheet in wb.sheets:
    
    charts = len(sheet.charts)
    shapes = len(sheet.shapes)

    if charts > 0 or shapes > 0:
        print(f"{sheet.name} contains charts/shapes")
    else:
        print(f"{sheet.name} contains only table data")

wb.close()
app.quit()

Approval contains only table data
1.0 contains charts/shapes
2A contains only table data
2B contains only table data
2C contains only table data
3-0  contains only table data
4-0 contains only table data
WP-HR-D contains charts/shapes
6A-CEO contains only table data
6B-COO contains only table data
6C-HR contains only table data
6D-PUR contains only table data
6E-QAD contains only table data
6F-PRD contains charts/shapes
6G-MAINT contains only table data
6H-DVP contains only table data
6I-QMS contains only table data
6J-PPC contains charts/shapes
6K-STORE contains charts/shapes
7A contains only table data
7B Obsolete contains only table data
7B New contains only table data
8 contains only table data
9-1 contains only table data
10A contains charts/shapes
10 B contains charts/shapes
CSR Rev 1 contains only table data
11 contains only table data
12 contains only table data


In [7]:
#Now we are extracting the text from those shapes
file = "A. IATF16949-QUALITY-MANUAL updated 30.9.25.xls"

app = xw.App(visible=False)
wb = app.books.open(file)

for sheet in wb.sheets:

    print(f"\n--- Sheet: {sheet.name} ---")

    # Extract text from shapes (text boxes, flowchart elements, etc.)
    for shape in sheet.shapes:
        try:
            text = shape.api.TextFrame2.TextRange.Text
            if text.strip():
                print("Shape text:", text)
        except:
            pass

    # Extract text from charts
    for chart in sheet.charts:
        try:
            title = chart.api.ChartTitle.Text
            print("Chart title:", title)
        except:
            pass

wb.close()
app.quit()


--- Sheet: Approval ---

--- Sheet: 1.0 ---

--- Sheet: 2A ---

--- Sheet: 2B ---

--- Sheet: 2C ---

--- Sheet: 3-0  ---

--- Sheet: 4-0 ---

--- Sheet: WP-HR-D ---
Shape text: Accounts and Excise(GST)
Vivek Bulakh
Mohan Shinde
Shape text: Maintenance
Sunil Gaikar
Shape text: Human Resources
Manoj Patil
Shape text: Purchase and Stores
Amit Gokhale
Shape text: Dispatch
Krishna Mohite
Shape text: Quality Assurance
Sanket Salunke
Shape text: Production (Please refer to WP-HRD-xx)
 
Shape text: Development & Costing  Vithal Jagtap             
Shape text: Regions
Shape text: Lucknow
Shyam  Paturkar
Shape text: Jamshedpur
Virendra Kharat
Shape text: Information Technology
Dheeraj Kumar
Shape text: Kanchan Pant
Chief Operating Officer
Shape text: Prabhakar Modak
Proprietor
Shape text: Vendor Quality Assurance
  Ganesh Deshpande
Shape text: Welding Excellence
  Gireesan Pokkali

--- Sheet: 6A-CEO ---

--- Sheet: 6B-COO ---

--- Sheet: 6C-HR ---

--- Sheet: 6D-PUR ---

--- Sheet: 6E-QAD ---

In [8]:
# A code to clean text, so we dont get bars when the code comes across empty cells
import re

def clean_excel_text(text):

    # Remove repeated pipes from empty cells
    text = re.sub(r'\|\s*\|+', '|', text)

    # Remove pipes at start or end
    text = re.sub(r'^\s*\|\s*', '', text)
    text = re.sub(r'\s*\|\s*$', '', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove very short noisy tokens (like single letters)
    text = re.sub(r'\b[a-zA-Z]\b', '', text)

    # Remove repeated separators
    text = re.sub(r'\s*\|\s*', ' ', text)

    # Remove extra whitespace again
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [9]:
# Rename file to remove the double dot
old_path = os.path.abspath(file)
new_path = old_path.replace("Deptt..xlsx", "Deptt.xlsx")

os.rename(old_path, new_path)

# Now work with the renamed file
app = xw.App(visible=False)

try:
    sheets = pd.read_excel(new_path, sheet_name=None)
    wb = app.books.open(new_path)

    for sheet_name, df in sheets.items():
        pass  # your logic here

finally:
    wb.close()
    app.quit()
    # Optionally rename back
    os.rename(new_path, old_path)


In [10]:
#A full code to extract cleaned text
documents = []
doc_id = 0

app = xw.App(visible=False)

for file in files:

    print(f"Processing file: {file}")

    # Load all table data
    sheets = pd.read_excel(file, sheet_name=None)

    # Open Excel to access shapes/charts
    wb = app.books.open(file)

    for sheet_name, df in sheets.items():

        doc_id += 1

        # -------------------------
        # 1. TABLE TEXT
        # -------------------------
        table_text = ""
        try:
            df = df.fillna("").astype(str)
            table_text = "\n".join(df.agg(" | ".join, axis=1))
        except:
            pass

        # -------------------------
        # 2. SHAPES / CHARTS TEXT
        # -------------------------
        shape_texts = []

        try:
            sheet = wb.sheets[sheet_name]

            # Shapes (flowcharts, textboxes)
            for shape in sheet.shapes:
                try:
                    text = shape.api.TextFrame2.TextRange.Text
                    if text and text.strip():
                        shape_texts.append(text.strip())
                except:
                    pass

            # Charts
            for chart in sheet.charts:
                try:
                    title = chart.api.ChartTitle.Text
                    if title and title.strip():
                        shape_texts.append(title.strip())
                except:
                    pass

        except:
            pass

        shape_text = "\n".join(shape_texts)

        # -------------------------
        # 3. COMBINE TEXT
        # -------------------------
        full_text = table_text + "\n" + shape_text

        # -------------------------
        # 4. CLEAN TEXT
        # -------------------------
        full_text = clean_excel_text(full_text)

        # Skip empty documents
        if len(full_text) == 0:
            continue

        # -------------------------
        # 5. STORE DOCUMENT + METADATA
        # -------------------------
        documents.append({
            "id": doc_id,
            "text": full_text,
            "metadata": {
                "file_name": file,
                "sheet_name": sheet_name,
                "source": "table + shapes/charts",
                "text_length": len(full_text)
            }
        })

    wb.close()

app.quit()

print("Total documents created:", len(documents))

Processing file: A. IATF16949-QUALITY-MANUAL updated 30.9.25.xls
Processing file: D.Turtle Chart_All Deptt..xlsx
Processing file: HR-Work Procedure Updated Issue 8.xlsx
Processing file: MAINT Work Procedure Issue 8.xls
Processing file: P .List of Customers & CSR matrix.xlsx
Processing file: Purchase Work Procedure Issue 8.xlsx
Processing file: Store Work Procedure Issue 8.xls
Processing file: WP Development .Issue 08.xls
Processing file: WP Production  Issue 8 (2).xls
Processing file: WP QAD updated issue 8.xls
Processing file: WP QMS Issue 8.xls
Total documents created: 264


In [11]:
documents[:2]

[{'id': 1,
  'text': 'CORPORATE QUALITY MANUAL Sharada Industries recognizes its resposibility as manufacturer of sheet metal components and assemblies and its quality requirements and to this end has developed and documented quality management system. The quality management system complies with the international standard IATF16949:2016, Automotive Quality Management System Standard and Customer Specific Requirements. The quality management system shall be commonly referred to as QMS throughout the organization. This Quality Manual applies to sites of the organization where customer-specified parts, for production are manufatured, supporting functions, whether on-site or remote and throughtout the entire automotive supply chain. This Quality Manual Has Been Issued And Approved from QM-1.0 to QM 12.0 and documents the Quality Management System established in compliance to IATF16949:2016 (1st edition) Only the Corporate MR has the explicit and implied authority to review, revise, modify 

In [12]:
#Retrieval of text from specific sheet
file_name = "A. IATF16949-QUALITY-MANUAL updated 30.9.25.xls"
sheet_name = "10 B"

result = next(
    doc for doc in documents
    if doc["metadata"]["file_name"] == file_name
    and doc["metadata"]["sheet_name"] == sheet_name
)

print(result["text"])

Document No. : QM-10.0B Interaction of Processes. Page: 01/01 Issue No. : 8 Issue Date: 30/06/2022 Rev No. : 00 Rev. Date: 30/06/2022 11.0 Prepared By: Verification of Corrective Actions Corrective Actions Business planning & Management Review 1.Business Planning 2. Roles, Resp, Authorities 3. Deployment of Quality Policy 4. Statutory and Regulatory Compliance 5. Risk Analysis 6. Preventive Controls 7. Corporate Responsibilities 8. Management Review 1. QMS Audit 2. Mfg. Process Audit 3. Product Audit MSA Calibration SUPPLIER Non-conforming product, processes and systems Customer Complaint and Resolution Effectiveness verification Production Dispatch Contingency Planning Customer Satisfaction CUSTOMER/ CSR Resources Maintenance Plating Heat treatment Powder Coating Spot/Projection Welding CED Coating CO2 Welding Corporate QMS / Internal Audit Production Planning Control & sales Production Production Planning Control TPM Preventive Predictive Breakdown SPC


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def chunk_text(text, chunk_size=500, overlap=100):  # 👈 500 instead of 50
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunked_data = []
for item in documents:
    text = item['text']
    chunks = chunk_text(text, chunk_size=500, overlap=100)
    for i, chunk in enumerate(chunks):
        vector = model.encode(chunk).tolist()
        chunked_data.append({
            'id': f"{item['id']}_chunk_{i}",
            'text': chunk,
            'vector': vector,
            'metadata': item['metadata']
        })

print(f"Total chunks created: {len(chunked_data)}")
print(chunked_data[0])  # preview first chunk

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks created: 2050
{'id': '1_chunk_0', 'text': 'CORPORATE QUALITY MANUAL Sharada Industries recognizes its resposibility as manufacturer of sheet metal components and assemblies and its quality requirements and to this end has developed and documented quality management system. The quality management system complies with the international standard IATF16949:2016, Automotive Quality Management System Standard and Customer Specific Requirements. The quality management system shall be commonly referred to as QMS throughout the organization. This', 'vector': [0.0251163337379694, -0.03033248893916607, -0.03028031252324581, -0.050815682858228683, -0.09988848865032196, -0.025215283036231995, 0.022079961374402046, -0.00994997937232256, -0.045577436685562134, 0.022161757573485374, 0.05113833025097847, -0.019670164212584496, 0.07518932968378067, 0.003084477735683322, -0.05993594229221344, -0.030422015115618706, 0.08051774650812149, -0.014391285367310047, 0.004930154886096716, -0.01574796

In [14]:
texts = [chunk['text'] for chunk in chunked_data]

In [15]:
embeddings = model.encode(texts, show_progress_bar=True)
print(embeddings.shape)

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

(2050, 384)


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
def semantic_search(query, chunked_data, embeddings, model, top_k=3):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = scores.argsort()[-top_k:][::-1]
    
    results = []
    for i in top_indices:
        results.append({
            'score': round(float(scores[i]), 4),
            'chunk_text': chunked_data[i]['text'],
            'metadata': chunked_data[i]['metadata']
        })
    return results

In [18]:
query = "Process Audit"
results = semantic_search(query, chunked_data, embeddings, model, top_k=3)

for r in results:
    print(f"Score: {r['score']}")
    print(f"Chunk: {r['chunk_text']}")
    print(f"File: {r['metadata']['file_name']} | Sheet: {r['metadata']['sheet_name']}")
    print("---")

Score: 0.6425
Chunk: Process approach auditing guidelines 2.7 Ensure that the Automotive Process approach is adhered to during the entire audit process. MR Process approach based observation sheets 2.8 Ensure that the Turtle charts for each Business process are properly understood by the auditors and auditee and are deployed during the audit process. MR Turtle charts from 1 to 16 3 Seek the guidance/help of qualified & experienced external auditor for internal audits in the following situations : MR 3.1 When ever th
File: WP QMS Issue 8.xls | Sheet: WP-QMS-1 Obsolete 
---
Score: 0.6311
Chunk: r auditing these processes 2.4.1. Work procedures to be referred for auditing these processes. CHT-QMS-7A / CHT-QMS-7B 2.4.1. Clauses to be audited in the Process 2.5 Automotive PROCESS-APPROACH MR 2.6 Ensure that the Automotive Process approach and auditing guidelines are properly understood by the auditors and auditee . MR Process approach auditing guidelines 2.7 Ensure that the Automotive Proce

In [19]:
#Building streamlit app
import pickle
import numpy as np

texts = [chunk['text'] for chunk in chunked_data]
embeddings = model.encode(texts, show_progress_bar=True)

with open("chunked_data.pkl", "wb") as f:
    pickle.dump(chunked_data, f)
np.save("embeddings.npy", embeddings)
print(f"Done! Total chunks: {len(chunked_data)}")

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

Done! Total chunks: 2050
